# 基本处理 #
- seasons_raw_data.csv 含有完整数据，但由于scrapy爬取的并发性质，数据没有按照系列名称分组。并且为了分析方便，希望把各系列按照评分人数从高到低排序。
- 因此在下一个cell中，将数据按照系列名称分组，组内按照“第一季，第二季...”排序，并将各系列按照评分人数降序排列。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#导入完整的数据
df = pd.read_csv('../data/raw/seasons_raw_data.csv')

#分组依据系列名称
group_by_column = 'series_name'
#组之间排序依据评分人数
group_sort_column = 'rate_num'
#组内排序依据季数（第一季、第二季...）
within_group_sort_column = 'season_num'

#对DataFrame进行分组
grouped_data = df.groupby(group_by_column)
# 提取每个分组中"rate_num"列的第一个值用于后续分组维度的排序，评分人数多的靠前排
group_first_rate_values = grouped_data[group_sort_column].first()
# 让同一分组的所有行都拥有该分组的rate_num基准值，方便后续整体排序
df['group_sort_temp'] = df[group_by_column].map(group_first_rate_values)
# 按临时列（分组的rate_num）降序排列，按季数升序排列
sorted_data = df.sort_values(
    by=['group_sort_temp', within_group_sort_column],
    ascending=[False, True]
)
# 删除临时列
final_sorted_data = sorted_data.drop(columns='group_sort_temp')

#导出
final_sorted_data.to_csv('../data/processed/grouped_and_sorted.csv', index=False)

# （可选）数据过滤 #
- 本项目初衷是探究长篇动漫的评分随季数的变化，所以最好只保留季数>=3的系列；
- 同时为了让数据更有研究意义，最好过滤掉评论数过少的系列。
- 当然这部分处理是可选的，可以按需删除、保留、修改或添加过滤条件 ^v^

In [ ]:
df = pd.read_csv('../data/processed/grouped_and_sorted.csv')

# 步骤1：筛选出「季数≥3」的动漫系列
series_season_count = df.groupby("series_name")["season_num"].nunique()  
valid_series_by_season = series_season_count[series_season_count >= 3].index  
df_step1 = df[df["series_name"].isin(valid_series_by_season)]  

# 步骤2：筛选出「该系列所有季的评分人数≥100」的动漫系列
series_rate_num_min = df_step1.groupby("series_name")["rate_num"].min()
valid_series_by_rate = series_rate_num_min[series_rate_num_min >= 1000].index  
df_cleaned = df_step1[df_step1["series_name"].isin(valid_series_by_rate)]

# for group in df_cleaned.groupby('series_name'):
#     print(group)

print(len(df_cleaned), len(df_cleaned.groupby('series_name')))

df_cleaned.to_csv('../data/processed/filtered_data.csv', index=False, encoding='utf-8-sig')